# 5 Post-Reconciliation Coordinate Maps
Visualise coordinates after the distance reconciliation step. Creates maps
with lines between sources filtered at different thresholds (0.1 km, 2 miles).

Input: data/1_derived/5_geocode_truck_stops/2_cross_source_distances.csv
Output: output/2_analysis/figures/coordinate_qc/ (HTML maps)

**Note:** This notebook reads a ~750MB CSV and builds large plotly map figures.
It may fail with `[Errno 12] Cannot allocate memory` on machines with limited
RAM (observed at ~2 GB). Run on a machine with at least 4 GB of free RAM.

In [ ]:
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from math import radians, cos, sin, asin, sqrt, atan2


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'data').exists() and (candidate / 'source').exists() and (candidate / 'temp').exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root(Path.cwd())
IN_FILE = PROJECT_ROOT / 'data' / '1_derived' / '5_geocode_truck_stops' / '2_cross_source_distances.csv'
FIG_DIR = PROJECT_ROOT / 'output' / '2_analysis' / 'figures' / 'coordinate_qc'
FIG_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(IN_FILE, low_memory=False)
print(f'Loaded {len(df):,} rows')

columns_info = [
    ('Webscraped_Phone_Latitude', 'Webscraped_Phone_Longitude', 'Truckstopsandservices (Phone)'),
    ('Webscraped_PlacedMatched_Latitude', 'Webscraped_PlacedMatched_Longitude', 'Truckstopsandservices (PlaceMatched)'),
    ('Yelp_Latitude', 'Yelp_Longitude', 'Yelp (Phone)'),
    ('YellowPages_JSONLD_LAT_1', 'YellowPages_JSONLD_LNG_1', 'YellowPages (Phone)')
]

def parse_coords(lat_str, lon_str):
    exclude = {'nan', '', 'none', 'removed'}
    lats = [float(x) for x in str(lat_str).split(';') if x.strip().lower() not in exclude]
    lons = [float(x) for x in str(lon_str).split(';') if x.strip().lower() not in exclude]
    return list(zip(lats, lons))

In [ ]:
# Scatter map of all sources (post-reconciliation)
latitudes, longitudes, sources = [], [], []

def extract(df, lat_col, lon_col, label):
    for _, row in df.iterrows():
        coords = parse_coords(row.get(lat_col, ''), row.get(lon_col, ''))
        for lat, lon in coords:
            latitudes.append(lat)
            longitudes.append(lon)
            sources.append(label)

for lat_col, lon_col, label in columns_info:
    extract(df, lat_col, lon_col, label)

plot_df = pd.DataFrame({'Latitude': latitudes, 'Longitude': longitudes, 'Source': sources})
fig = px.scatter_mapbox(plot_df, lat='Latitude', lon='Longitude', color='Source',
                        zoom=3, height=900, width=1800)
fig.update_layout(mapbox_style='open-street-map', margin={'r':0,'t':0,'l':0,'b':0})
fig.show()
fig.write_html(str(FIG_DIR / '5_post_reconciliation_sources.html'))

In [ ]:
# Disagreement map at 2-mile threshold (post-reconciliation)
fig = go.Figure()
min_dist_threshold = 2  # miles

for idx, row in df.iterrows():
    all_coords = []
    hover_texts = []
    for lat_col, lon_col, label in columns_info:
        coords = parse_coords(row.get(lat_col, ''), row.get(lon_col, ''))
        if coords:
            all_coords.extend(coords)
            hover_texts.extend([f'Row: {idx}<br>Source: {label}'] * len(coords))
    show = False
    if len(all_coords) > 1:
        for i in range(1, len(all_coords)):
            lat1, lon1 = all_coords[i - 1]
            lat2, lon2 = all_coords[i]
            R = 3958.8
            dlat = radians(lat2 - lat1)
            dlon = radians(lon2 - lon1)
            a = sin(dlat / 2) ** 2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon / 2) ** 2
            d = R * 2 * asin(sqrt(a))
            if d >= min_dist_threshold:
                show = True
                break
    if show and len(all_coords) > 1:
        lats, lons = zip(*all_coords)
        fig.add_trace(go.Scattermapbox(
            lat=lats, lon=lons, mode='markers+lines',
            marker={'size': 8}, line={'width': 2},
            name=f'Row {idx}', text=hover_texts, hoverinfo='text+lat+lon'
        ))

fig.update_layout(
    mapbox_style='open-street-map', mapbox_zoom=4,
    mapbox_center={'lat': 39.5, 'lon': -98.35},
    margin={'r': 0, 't': 0, 'l': 0, 'b': 0}, showlegend=True
)
fig.show()
fig.write_html(str(FIG_DIR / '5_post_reconciliation_disagreements_2mi.html'))
print('Maps saved.')